# ETL Exploration
Use this notebook to prototype and test your ETL pipeline before moving code to `etl/etl.py`.

In [1]:
import pandas as pd
import numpy as np
import os
from dotenv import load_dotenv

load_dotenv()
SIMFIN_API_KEY = os.getenv('SIMFIN_API_KEY')
print('API key loaded:', bool(SIMFIN_API_KEY))

API key loaded: False


## 1. Fetch raw data

In [2]:
# TODO
df = pd.read_csv('../data/raw/us-shareprices-daily.csv', sep=';')

## 2. Clean & explore
Check for nulls, data types, date ranges.

In [3]:
df.shape

(6206232, 11)

In [4]:
df.dtypes

Ticker                 object
SimFinId                int64
Date                   object
Open                  float64
High                  float64
Low                   float64
Close                 float64
Adj. Close            float64
Volume                  int64
Dividend              float64
Shares Outstanding    float64
dtype: object

In [5]:
df.isnull().sum()

Ticker                    668
SimFinId                    0
Date                        0
Open                        0
High                        0
Low                         0
Close                       0
Adj. Close                  0
Volume                      0
Dividend              6169933
Shares Outstanding     525204
dtype: int64

### How are we handling missing values?
1. We are going to apply fillna(0) to Dividend
2. Not changing Shares outstanding as we are not going to use it as a feature. 
3. In regards to Ticker, it won't affect anything since we are going to filter by specific tickets.

In [6]:
#Check date range
print(f'Min date: {df.Date.min()}, Max date: {df.Date.max()}')

Min date: 2020-04-06, Max date: 2025-03-10


In [7]:
companies_df = pd.read_csv('../data/raw/us-companies.csv', sep=';')
companies_df.head()

,Ticker,SimFinId,Company Name,IndustryId,ISIN,End of financial year (month),Number Employees,Business Summary,Market,CIK,Main Currency
0,NaN,20095034,NaN,NaN,NaN,NaN,NaN,NaN,us,1986247.0,USD
1,NaN,18692750,NaN,NaN,NaN,NaN,NaN,NaN,us,1997711.0,USD
2,NaN,18847915,NaN,NaN,NaN,NaN,NaN,NaN,us,1769731.0,USD
3,NaN,18538670,NaN,NaN,NaN,NaN,NaN,NaN,us,1734107.0,USD
4,NaN,18657366,NaN,NaN,NaN,NaN,NaN,NaN,us,1899830.0,USD


In [8]:
companies_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6529 entries, 0 to 6528
Data columns (total 11 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Ticker                         6491 non-null   object 
 1   SimFinId                       6529 non-null   int64  
 2   Company Name                   6494 non-null   object 
 3   IndustryId                     6225 non-null   float64
 4   ISIN                           5345 non-null   object 
 5   End of financial year (month)  6495 non-null   float64
 6   Number Employees               5703 non-null   float64
 7   Business Summary               6234 non-null   object 
 8   Market                         6529 non-null   object 
 9   CIK                            6517 non-null   float64
 10  Main Currency                  6529 non-null   object 
dtypes: float64(4), int64(1), object(6)
memory usage: 561.2+ KB


## Building Data Cleansing Functions

**Why?** We want to avoid redundancy and being more time-efficient

In [9]:
TICKERS = [
    'AAPL', 'MSFT', 'GOOG', 'META', 'NVDA',
    'ORCL', 'ADBE', 'CRM', 'INTC', 'AMD',
    'AMZN', 'WMT', 'MCD', 'DIS', 'KO',
    'JPM', 'BAC', 'GS', 'V', 'MA',
    'JNJ', 'PFE', 'UNH', 'AVGO', 'QCOM',
    'TSLA', 'NFLX', 'PLTR', 'COST', 'PEP',
]

def clean(df):
    df = df.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date').reset_index(drop=True)
    df['Dividend'] = df['Dividend'].fillna(0)
    df = df.drop(columns=['SimFinId'])

    # Detect price errors: daily returns above 50% are almost certainly data errors
    # or unadjusted stock splits. Instead of dropping the row (which creates a phantom
    # multi-day return in the next pct_change()), we set Adj. Close to NaN and forward-fill.
    # This preserves time-series continuity -- all rows stay intact with no gaps,
    # so Return, Return_Lag1, Return_Lag2 are never corrupted by artificial multi-day spans.
    daily_return = df['Adj. Close'].pct_change()
    mask = daily_return.abs() > 0.5
    n_errors = int(mask.sum())
    if n_errors > 0:
        print(f'  Warning: Detected {n_errors} price error(s) with |return| > 50% -- nullified and forward-filled.')
        df.loc[mask, 'Adj. Close'] = np.nan
        df['Adj. Close'] = df['Adj. Close'].ffill()

    return df

## 3. Feature engineering
Create the derived columns the model will use.

In [10]:
def engineer_features(df):
    df = df.copy()

    # NOTE: This prototype uses Close (not Adj. Close).
    # The canonical ETL (etl/etl.py) uses Adj. Close — same logic applies.
    price = df['Close']

    # Daily percentage return
    df['Return'] = price.pct_change()

    # Moving averages: smooth out noise to reveal the underlying trend
    df['MA5'] = price.rolling(5).mean()   # short-term (1 week)
    df['MA20'] = price.rolling(20).mean() # medium-term (1 month)

    # Volume change: detects unusual trading activity
    df['Volume_Change'] = df['Volume'].pct_change()

    # Winsorize Return and Volume_Change at the 1st / 99th percentile.
    # Bounds outlier influence on lag features while keeping all rows in the dataset.
    for col in ('Return', 'Volume_Change'):
        lo = df[col].quantile(0.01)
        hi = df[col].quantile(0.99)
        df[col] = df[col].clip(lo, hi)

    # Market Cap: total market value = price x shares outstanding
    df['Market_Cap'] = price * df['Shares Outstanding']

    # Target: next-day return (continuous regression target)
    df['Target'] = price.pct_change().shift(-1)

    # RSI (Relative Strength Index): >70 = overbought, <30 = oversold
    delta = price.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    df['RSI'] = 100 - (100 / (1 + gain / loss))

    # MACD: trend direction and momentum
    ema12 = price.ewm(span=12, adjust=False).mean()
    ema26 = price.ewm(span=26, adjust=False).mean()
    df['MACD'] = ema12 - ema26
    df['MACD_Signal'] = df['MACD'].ewm(span=9, adjust=False).mean()

    # Bollinger Bands: volatility indicator
    std20 = price.rolling(20).std()
    df['BB_Upper'] = df['MA20'] + 2 * std20
    df['BB_Lower'] = df['MA20'] - 2 * std20

    # Lag features: give the model memory of recent returns
    df['Return_Lag1'] = df['Return'].shift(1)  # yesterday's return
    df['Return_Lag2'] = df['Return'].shift(2)  # 2 days ago

    # ── Volatility-normalised features for cross-stock comparability ──────
    # Stocks have very different daily volatility:
    #   KO swings ~0.5%/day, TSLA swings ~3%/day, NVDA ~3.3%/day.
    # Training a pooled model on raw arithmetic returns lets high-vol stocks
    # dominate the loss function. Dividing by realised volatility puts every
    # ticker on the same risk-adjusted scale.

    # Log return: log(P_t / P_{t-1}). Additive over time, more Gaussian
    # than arithmetic returns, and correct for multi-period compounding.
    import numpy as np
    df['Log_Return'] = np.log(price / price.shift(1))

    # 20-day rolling realised volatility: std of daily log returns
    # over the past month -- captures the current volatility regime.
    df['Volatility_20'] = df['Log_Return'].rolling(20).std()

    # Volatility-normalised return: log return / recent volatility.
    # Analogous to a daily Z-score (value of +1 = rose one std dev today).
    # Directly comparable across KO, TSLA, NVDA, etc.
    df['Return_norm'] = df['Log_Return'] / df['Volatility_20']

    # Lagged normalised returns: memory of recent risk-adjusted moves
    df['Return_norm_Lag1'] = df['Return_norm'].shift(1)
    df['Return_norm_Lag2'] = df['Return_norm'].shift(2)

    # Drop rows with NaN in price-derived columns (rolling windows + shifts).
    # Fundamental columns are excluded: they can legitimately be NaN for
    # some tickers (e.g. banks, payment networks) and are handled downstream.
    price_required = [
        'Return', 'MA5', 'MA20', 'Volume_Change', 'Market_Cap', 'Target',
        'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
        'Return_Lag1', 'Return_Lag2',
        'Log_Return', 'Volatility_20', 'Return_norm',
        'Return_norm_Lag1', 'Return_norm_Lag2',
    ]
    df = df.dropna(subset=price_required).reset_index(drop=True)
    return df

processed = {}
for ticker in TICKERS:
    company_df = df[df['Ticker'] == ticker]
    cleaned = clean(company_df)
    featured = engineer_features(cleaned)
    featured.to_csv(f'../data/processed/{ticker}.csv', index=False)
    processed[ticker] = featured
    print(f'{ticker}: {len(featured)} rows, {len(featured.columns)} columns saved')


AAPL: 1215 rows, 28 columns saved
MSFT: 1215 rows, 28 columns saved
GOOG: 1215 rows, 28 columns saved
META: 1215 rows, 28 columns saved
NVDA: 1215 rows, 28 columns saved
ORCL: 1215 rows, 28 columns saved
ADBE: 1215 rows, 28 columns saved
CRM: 1215 rows, 28 columns saved
INTC: 1215 rows, 28 columns saved
AMD: 1215 rows, 28 columns saved
AMZN: 1215 rows, 28 columns saved
WMT: 1215 rows, 28 columns saved
MCD: 1215 rows, 28 columns saved
DIS: 1215 rows, 28 columns saved
KO: 1215 rows, 28 columns saved
JPM: 1215 rows, 28 columns saved
BAC: 1215 rows, 28 columns saved
GS: 1215 rows, 28 columns saved
V: 1215 rows, 28 columns saved
MA: 1215 rows, 28 columns saved
JNJ: 1215 rows, 28 columns saved
PFE: 1215 rows, 28 columns saved
UNH: 1215 rows, 28 columns saved
AVGO: 1215 rows, 28 columns saved
QCOM: 1215 rows, 28 columns saved
TSLA: 1215 rows, 28 columns saved
NFLX: 1215 rows, 28 columns saved
PLTR: 1092 rows, 28 columns saved
COST: 1215 rows, 28 columns saved
PEP: 1215 rows, 28 columns saved


## 4. Fundamental Data Enrichment

We enrich daily price data with quarterly financial ratios from income statements, balance sheets, and cash flow statements.

**Key challenge — look-ahead bias:** quarterly reports are published weeks *after* the fiscal period ends. We use `Publish Date` (not `Fiscal Year`) so we only attach data that was publicly available on each trading day.

**Technique:** `pd.merge_asof` with `direction='backward'` — for each trading day, attach the most recently *published* quarterly snapshot. This guarantees zero look-ahead bias.

| Source | Raw columns used | Derived feature | Meaning |
|---|---|---|---|
| Income | Gross Profit / Revenue | `Gross_Margin` | Operational efficiency |
| Income | Operating Income / Revenue | `Operating_Margin` | Core profitability |
| Income | Net Income / Revenue | `Net_Margin` | Bottom-line profitability |
| Balance | Total Liabilities / Total Equity | `Debt_to_Equity` | Financial leverage / risk |
| Cashflow + Balance | Operating CF / Total Assets | `Operating_CF_Ratio` | Cash generation quality |

In [11]:
# Load only the columns we need from each quarterly file
income_q = pd.read_csv('../data/raw/us-income-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Revenue', 'Gross Profit',
             'Operating Income (Loss)', 'Net Income'])

balance_q = pd.read_csv('../data/raw/us-balance-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Total Assets', 'Total Liabilities', 'Total Equity'])

cashflow_q = pd.read_csv('../data/raw/us-cashflow-quarterly.csv', sep=';',
    usecols=['Ticker', 'Publish Date', 'Net Cash from Operating Activities'])

# Filter to our 5 tickers only
income_q   = income_q[income_q['Ticker'].isin(TICKERS)].copy()
balance_q  = balance_q[balance_q['Ticker'].isin(TICKERS)].copy()
cashflow_q = cashflow_q[cashflow_q['Ticker'].isin(TICKERS)].copy()

for name, frame in [('Income', income_q), ('Balance', balance_q), ('Cashflow', cashflow_q)]:
    n_quarters = frame[frame['Ticker'] == 'AAPL']['Publish Date'].nunique()
    print(f"{name:10s} — {len(frame):4d} rows | AAPL: {n_quarters} quarters")

print()
income_q[income_q['Ticker'] == 'AAPL'][['Ticker', 'Publish Date', 'Revenue', 'Gross Profit', 'Net Income']].tail(4)

Income     —  519 rows | AAPL: 19 quarters
Balance    —  521 rows | AAPL: 19 quarters
Cashflow   —  520 rows | AAPL: 19 quarters



,Ticker,Publish Date,Revenue,Gross Profit,Net Income
142,AAPL,2024-05-03,9.075300e+10,4.227100e+10,23636000000
143,AAPL,2024-08-02,8.577700e+10,3.967800e+10,21448000000
144,AAPL,2024-11-01,9.493000e+10,4.387900e+10,14736000000
145,AAPL,2025-01-31,1.243000e+11,5.827500e+10,36330000000


In [12]:
FUNDAMENTAL_FEATURES = ['Gross_Margin', 'Operating_Margin', 'Net_Margin',
                        'Debt_to_Equity', 'Operating_CF_Ratio']

def compute_fundamental_features(income, balance, cashflow):
    """
    Compute normalized financial ratios from raw quarterly statements.
    Dimensionless ratios are comparable across companies of different sizes.
    """
    # --- Income ratios ---
    inc = income.copy()
    inc['Gross_Margin']     = inc['Gross Profit'] / inc['Revenue']
    inc['Operating_Margin'] = inc['Operating Income (Loss)'] / inc['Revenue']
    inc['Net_Margin']       = inc['Net Income'] / inc['Revenue']
    inc = inc[['Ticker', 'Publish Date', 'Gross_Margin', 'Operating_Margin', 'Net_Margin']]

    # --- Balance ratio ---
    bal = balance.copy()
    # Debt-to-Equity: how much of the company is financed by debt vs equity.
    # High D/E = higher financial risk. Can be negative if equity < 0 (distressed).
    bal['Debt_to_Equity'] = bal['Total Liabilities'] / bal['Total Equity']
    bal_ratios = bal[['Ticker', 'Publish Date', 'Debt_to_Equity']]

    # --- Cash flow ratio (normalised by total assets) ---
    # Operating CF Ratio: cash generated from core operations relative to asset base.
    # More reliable signal than net income — harder to manipulate with accounting.
    cf = cashflow.merge(
        bal[['Ticker', 'Publish Date', 'Total Assets']],
        on=['Ticker', 'Publish Date'], how='left'
    )
    cf['Operating_CF_Ratio'] = cf['Net Cash from Operating Activities'] / cf['Total Assets']
    cf_ratios = cf[['Ticker', 'Publish Date', 'Operating_CF_Ratio']]

    # --- Combine all ---
    fund = inc.merge(bal_ratios, on=['Ticker', 'Publish Date'], how='outer')
    fund = fund.merge(cf_ratios,  on=['Ticker', 'Publish Date'], how='outer')
    fund['Publish Date'] = pd.to_datetime(fund['Publish Date'])
    # Replace inf (e.g. Revenue=0, or Equity=0) with NaN so dropna handles it cleanly
    fund = fund.replace([np.inf, -np.inf], np.nan)
    fund = fund.sort_values(['Ticker', 'Publish Date']).reset_index(drop=True)
    return fund

fundamentals = compute_fundamental_features(income_q, balance_q, cashflow_q)

print(f"Fundamentals shape: {fundamentals.shape}")
print(f"\nSample — AAPL recent quarters:")
fundamentals[fundamentals['Ticker'] == 'AAPL'][['Publish Date'] + FUNDAMENTAL_FEATURES].tail(6)

Fundamentals shape: (538, 7)

Sample — AAPL recent quarters:


,Publish Date,Gross_Margin,Operating_Margin,Net_Margin,Debt_to_Equity,Operating_CF_Ratio
13,2023-11-03,0.451708,0.301336,0.256497,4.673462,0.061256
14,2024-02-02,0.458750,0.337637,0.283638,3.770769,0.112853
15,2024-05-03,0.465781,0.307428,0.260443,3.547686,0.067247
16,2024-08-02,0.462572,0.295557,0.250044,3.971098,0.087023
17,2024-11-01,0.462225,0.311714,0.155230,5.408780,0.073459
18,2025-01-31,0.468825,0.344586,0.292277,4.154214,0.086999


In [13]:
def merge_fundamentals_for_ticker(price_df, fundamentals, ticker):
    """
    Point-in-time merge for a single ticker.

    For each daily trading row, attaches the most recently PUBLISHED quarterly data.
    merge_asof with direction='backward' ensures we never peek into the future:
    if today is 2024-02-01 and the last earnings were published 2024-01-30, those
    values are used. The next quarter's report (e.g. 2024-04-30) is not visible yet.
    """
    p = price_df.sort_values('Date')
    f = (fundamentals[fundamentals['Ticker'] == ticker]
         .drop(columns='Ticker')
         .sort_values('Publish Date'))

    return pd.merge_asof(
        p, f,
        left_on='Date',
        right_on='Publish Date',
        direction='backward'   # only use data published ON OR BEFORE the trading day
    )


# Re-run the full pipeline: clean → merge fundamentals → engineer features → save
PRICE_FEATURES = ['Return', 'MA5', 'MA20', 'Volume_Change', 'Market_Cap',
                  'RSI', 'MACD', 'MACD_Signal', 'BB_Upper', 'BB_Lower',
                  'Return_Lag1', 'Return_Lag2']
ALL_FEATURES = PRICE_FEATURES + FUNDAMENTAL_FEATURES

enriched = {}
for ticker in TICKERS:
    raw     = df[df['Ticker'] == ticker]
    cleaned = clean(raw)
    merged  = merge_fundamentals_for_ticker(cleaned, fundamentals, ticker)
    featured = engineer_features(merged)
    featured.to_csv(f'../data/processed/{ticker}.csv', index=False)
    enriched[ticker] = featured
    n_nan_fund = featured[FUNDAMENTAL_FEATURES].isna().sum().sum()
    print(f"{ticker}: {len(featured)} rows saved | NaN in fundamental features: {n_nan_fund}")

AAPL: 1215 rows saved | NaN in fundamental features: 295
MSFT: 1215 rows saved | NaN in fundamental features: 290
GOOG: 1215 rows saved | NaN in fundamental features: 295
META: 1215 rows saved | NaN in fundamental features: 295
NVDA: 1215 rows saved | NaN in fundamental features: 50
ORCL: 1215 rows saved | NaN in fundamental features: 155
ADBE: 1215 rows saved | NaN in fundamental features: 507
CRM: 1215 rows saved | NaN in fundamental features: 80
INTC: 1215 rows saved | NaN in fundamental features: 270
AMD: 1215 rows saved | NaN in fundamental features: 285
AMZN: 1215 rows saved | NaN in fundamental features: 518
WMT: 1215 rows saved | NaN in fundamental features: 90
MCD: 1215 rows saved | NaN in fundamental features: 513
DIS: 1215 rows saved | NaN in fundamental features: 402
KO: 1215 rows saved | NaN in fundamental features: 260
JPM: 1215 rows saved | NaN in fundamental features: 6075
BAC: 1215 rows saved | NaN in fundamental features: 6075
GS: 1215 rows saved | NaN in fundamental 

In [14]:
# Complete feature inventory — what the ML model will see
sample = enriched['AAPL']

print("=" * 60)
print("COMPLETE FEATURE SET  (AAPL sample)")
print("=" * 60)
print(f"\n{'#':<4} {'Feature':<22} {'Type':<8} {'Min':>10}  {'Max':>10}  {'NaN':>5}")
print("-" * 60)

all_cols = ALL_FEATURES + ['Target']
for i, col in enumerate(all_cols, 1):
    s = sample[col]
    tag = "PRICE" if col in PRICE_FEATURES else ("FUND" if col in FUNDAMENTAL_FEATURES else "TARGET")
    print(f"{i:<4} {col:<22} {tag:<8} {s.min():>10.4f}  {s.max():>10.4f}  {s.isna().sum():>5}")

print(f"\nTotal trainable features : {len(ALL_FEATURES)}  ({len(PRICE_FEATURES)} price + {len(FUNDAMENTAL_FEATURES)} fundamental)")
print(f"Date range               : {sample['Date'].min().date()} → {sample['Date'].max().date()}")
print(f"Total rows (AAPL)        : {len(sample)}")
print(f"\nTarget (next-day return) : mean={sample['Target'].mean():.4f}  std={sample['Target'].std():.4f}"
      f"  range=[{sample['Target'].min():.4f}, {sample['Target'].max():.4f}]")

COMPLETE FEATURE SET  (AAPL sample)

#    Feature                Type            Min         Max    NaN
------------------------------------------------------------
1    Return                 PRICE       -0.0467      0.0463      0
2    MA5                    PRICE       74.2100    256.5140      0
3    MA20                   PRICE       71.1130    249.9850      0
4    Volume_Change          PRICE       -0.4950      1.1811      0
5    Market_Cap             PRICE    1316597599600.0000  3915300473459.9995      0
6    RSI                    PRICE        3.1429     96.1630      0
7    MACD                   PRICE       -6.6731      8.9481      0
8    MACD_Signal            PRICE       -5.8157      8.3104      0
9    BB_Upper               PRICE       76.1459    265.2736      0
10   BB_Lower               PRICE       65.9222    240.3220      0
11   Return_Lag1            PRICE       -0.0467      0.0463      0
12   Return_Lag2            PRICE       -0.0467      0.0463      0
13   Gross_Marg

In [15]:
enriched['AAPL'].head(20).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL,AAPL
Date,2020-05-07 00:00:00,2020-05-08 00:00:00,2020-05-11 00:00:00,2020-05-12 00:00:00,2020-05-13 00:00:00,2020-05-14 00:00:00,2020-05-15 00:00:00,2020-05-18 00:00:00,2020-05-19 00:00:00,2020-05-20 00:00:00,2020-05-21 00:00:00,2020-05-22 00:00:00,2020-05-26 00:00:00,2020-05-27 00:00:00,2020-05-28 00:00:00,2020-05-29 00:00:00,2020-06-01 00:00:00,2020-06-02 00:00:00,2020-06-03 00:00:00,2020-06-04 00:00:00
Open,75.81,76.41,77.03,79.46,78.04,76.13,75.09,78.29,78.76,79.17,79.67,78.94,80.88,79.03,79.19,79.81,79.44,80.19,81.17,81.1
High,76.29,77.59,79.26,79.92,78.99,77.45,76.97,79.12,79.63,79.88,80.22,79.81,81.06,79.68,80.86,80.29,80.59,80.86,81.55,81.41
Low,75.49,76.07,76.81,77.73,75.8,75.38,75.05,77.58,78.25,79.13,78.97,78.84,79.12,78.27,78.91,79.12,79.3,79.73,80.58,80.19
Close,75.94,77.53,78.75,77.85,76.91,77.39,76.93,78.74,78.28,79.81,79.21,79.72,79.18,79.53,79.56,79.48,80.46,80.83,81.28,80.58
Adj. Close,73.4,75.14,76.32,75.45,74.54,75.0,74.56,76.31,75.87,77.35,76.77,77.26,76.74,77.08,77.11,77.03,77.98,78.34,78.77,78.1
Volume,115215056,134047940,145946244,162301052,200622556,158929076,166348376,135372500,101729540,111504860,102688844,81803016,125521816,112945096,133796412,153598128,81018612,87642816,104491216,87560364
Dividend,0.0,0.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Shares Outstanding,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0,17337340000.0


## 5. Data Dictionary

### Group 1: Raw Price Columns *(reference only — not used as ML features)*

| Field | Type | Description |
|---|---|---|
| `Ticker` | string | Stock ticker symbol (e.g. `AAPL`) |
| `Date` | date | Trading day |
| `Open` | float | Opening price of the session |
| `High` | float | Highest price of the session |
| `Low` | float | Lowest price of the session |
| `Close` | float | Closing price (unadjusted) |
| `Adj. Close` | float | Closing price adjusted for splits and dividends — used for all price-based feature calculations in `etl.py` |
| `Volume` | int | Number of shares traded |
| `Dividend` | float | Dividend paid on that day (0 if none) |
| `Shares Outstanding` | float | Total shares outstanding |

---

### Group 2: Price-Based Model Features *(use in model)*

These 6 features are computed from daily price/volume data and are used directly as model inputs.

| Field | Type | Description |
|---|---|---|
| `MA5` | float | 5-day simple moving average of `Adj. Close`. Smooths short-term noise (≈ 1 week trend) |
| `MA20` | float | 20-day simple moving average of `Adj. Close`. Captures medium-term trend (≈ 1 month) |
| `Volume_Change` | float | Daily % change in volume. Spikes signal unusual market activity |
| `Market_Cap` | float | Total market value = `Adj. Close × Shares Outstanding`. Dynamic proxy for company size |
| `RSI` | float | Relative Strength Index (14-day). Momentum indicator: >70 = overbought, <30 = oversold |
| `MACD` | float | Difference between 12-day and 26-day EMA. Positive = bullish momentum, negative = bearish |

---

### Group 3: Volatility-Normalised Model Features *(used in model)*

These 5 features put every ticker on the same risk-adjusted scale, enabling fair cross-stock comparison.
Stocks have very different daily volatility (KO ±0.5%/day vs TSLA ±3%/day); dividing by realised volatility neutralises this.

| Field | Type | Formula | Description |
|---|---|---|---|
| `Log_Return` | float | log(Adj.Closeₜ / Adj.Closeₜ₋₁) | Additive daily return, more Gaussian than arithmetic return |
| `Volatility_20` | float | rolling(20).std(Log_Return) | 20-day realised volatility — captures current volatility regime |
| `Return_norm` | float | Log_Return / Volatility_20 | Risk-adjusted return (daily Z-score) — directly comparable across tickers |
| `Return_norm_Lag1` | float | Return_norm.shift(1) | Yesterday’s risk-adjusted return |
| `Return_norm_Lag2` | float | Return_norm.shift(2) | Risk-adjusted return from 2 days ago |

---

### Group 4: Fundamental Model Features *(used in model — quarterly, point-in-time merged)*

These 5 features come from quarterly filings and are merged with a point-in-time join (using `Publish Date`) to avoid look-ahead bias.

| Field | Type | Source file | Description |
|---|---|---|---|
| `Gross_Margin` | float | Income | Gross Profit / Revenue. How much revenue remains after production costs |
| `Operating_Margin` | float | Income | Operating Income / Revenue. Efficiency of core business operations |
| `Net_Margin` | float | Income | Net Income / Revenue. Bottom-line profitability after all expenses and taxes |
| `Debt_to_Equity` | float | Balance | Total Liabilities / Total Equity. Financial leverage — higher = more debt-dependent |
| `Operating_CF_Ratio` | float | Cashflow + Balance | Operating Cash Flow / Total Assets. Quality of earnings — harder to manipulate than net income |
| `Publish Date` | date | — | Date the quarterly report was published. Used for the point-in-time merge — not a feature |

**Note:** JPM, BAC, GS use a different income statement format → price-only (fundamentals will be NaN).
V and MA have no “Gross Profit” line → `Gross_Margin` is NaN; other fundamental ratios are present.

---

### Group 5: Display-Only Engineered Features *(used for Streamlit visualisation only)*

These 6 features are computed by the ETL and stored in the processed CSVs, but are **not fed to the ML model**.
They are read by the Streamlit app (`go_live.py`) to render charts and metrics.

| Field | Type | Used in Streamlit for | Description |
|---|---|---|---|
| `Return` | float | “Daily Return” metric widget | Daily % change in `Adj. Close` |
| `MACD_Signal` | float | MACD chart (signal line) | 9-day EMA of MACD. Used to draw the signal line and colour the histogram |
| `BB_Upper` | float | Price chart (Bollinger shading) | Upper Bollinger Band = MA20 + 2×std20. Price near here = overbought |
| `BB_Lower` | float | Price chart (Bollinger shading) | Lower Bollinger Band = MA20 − 2×std20. Price near here = oversold |
| `Return_Lag1` | float | Indicator values table | Yesterday’s arithmetic return |
| `Return_Lag2` | float | Indicator values table | Arithmetic return from 2 days ago |

---

### Group 6: Target *(what the model predicts)*

| Field | Type | Description |
|---|---|---|
| `Target` | float | **Next-day return**: `(price_tomorrow − price_today) / price_today`. Continuous regression target. Positive = price rises next day, negative = price falls. The web app converts this to a category and signal using fixed thresholds. |

**Category thresholds applied to predicted return:**

| Predicted return | Category | Trading signal |
|---|---|---|
| > +2.0% | HIGH RISE | BUY |
| +0.5% to +2.0% | LOW RISE | BUY |
| −0.5% to +0.5% | STAY | HOLD |
| −2.0% to −0.5% | LOW FALL | SELL |
| < −2.0% | HIGH FALL | SELL |

---

**Summary:** 16 model features (6 price-based + 5 volatility-normalised + 5 fundamental) + 6 display-only columns → 1 continuous regression target (next-day return).